# 🍬 GUMMY FORMULATION INTELLIGENCE PLATFORM
## ULTIMATE EDITION: 14-Phase Ultra-Granular Analytics

**Complete Analysis Pipeline:**
- 14 Phases of Comprehensive Analysis
- 45+ Publication-Quality Visualizations
- 30+ CSV Data Exports
- 8+ Statistical Tests
- 9 ML Algorithms
- Automatic ZIP Output

---

In [1]:
# SETUP: Install and Import Libraries
!pip install -q pandas numpy scipy scikit-learn matplotlib seaborn plotly statsmodels xgboost shap openpyxl lightgbm catboost kaleido pingouin

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime
import os
import zipfile
import shutil

from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, wilcoxon, shapiro, levene, f_oneway, anderson, kstest, normaltest, jarque_bera
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import xgboost as xgb
import lightgbm as lgb
import shap
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Setup directories
dirs = ['outputs', 'outputs/phase_01_discovery', 'outputs/phase_02_design', 'outputs/phase_03_eda',
        'outputs/phase_04_statistics', 'outputs/phase_05_multivariate', 'outputs/phase_08_ml',
        'outputs/phase_10_indices', 'outputs/reports']
for d in dirs:
    os.makedirs(d, exist_ok=True)

sns.set_style('whitegrid')
colors_4 = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
formulation_colors = {'A': '#FF6B6B', 'B': '#4ECDC4', 'C': '#45B7D1', 'D': '#FFA07A'}
formulation_map = {'A': 'Corn Flour', 'B': 'Oats Flour', 'C': 'Rice Flour', 'D': 'Puffed Rice Powder'}

print('✓ Environment initialized')
print(f'✓ Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.9 MB/s eta 0:00:00
✓ Environment initialized
✓ Timestamp: 2026-06-27 12:05:01


In [2]:
try:
    from google.colab import files
    print('📁 Upload gummies_data.xlsx...')
    uploaded = files.upload()
    excel_file = list(uploaded.keys())[0]
except:
    excel_file = 'gummies_data.xlsx'

print(f'✓ File: {excel_file}')

📁 Upload gummies_data.xlsx...


Saving gummies data.xlsx to gummies data.xlsx
✓ File: gummies data.xlsx


# PHASE 1: DATA DISCOVERY & INTEGRATION

In [3]:
print('\n' + '='*100)
print('PHASE 1: DATA DISCOVERY')
print('='*100)

xls = pd.ExcelFile(excel_file)
sheet_names = xls.sheet_names
raw_data = {}

print(f'\n{"Sheet Name":<35} {"Rows":>6} {"Cols":>6}')
print('-'*50)

for sheet in sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet)
    raw_data[sheet] = df
    print(f'{sheet:<35} {df.shape[0]:>6} {df.shape[1]:>6}')

print(f'\n✓ Total sheets: {len(raw_data)}')


PHASE 1: DATA DISCOVERY

Sheet Name                            Rows   Cols
--------------------------------------------------
Proximate Composition                   15      8
Minerals                                14     10
Vitamins                                15     12
Texture Analysis                        13      4
pH                                      13      2
Colour Profile Analysis                 13      4
Sensory Evaluation                      12      7
Phytochemical and Antioxidant           12      4

✓ Total sheets: 8


In [4]:
# Data Integration
print('\nINTEGRATING DATA...')

integrated = {}

prox = raw_data['Proximate Composition'].iloc[2:].reset_index(drop=True)
prox.columns = ['Formulation', 'Energy_kcal', 'Carb_pct', 'Protein_pct', 'Fat_pct', 'Moisture_g', 'Ash_g', 'Fiber_g']
prox = prox[prox['Formulation'].notna()]
for col in prox.columns[1:]:
    prox[col] = pd.to_numeric(prox[col], errors='coerce')
integrated['Proximate'] = prox

min_data = raw_data['Minerals'].iloc[2:].reset_index(drop=True)
min_data.columns = ['Formulation', 'Na_mg', 'K_mg', 'Ca_mg', 'Zn_mg', 'Fe_mg', 'P_mg', 'I_mg', 'Mg_mg', 'Cu_mg']
min_data = min_data[min_data['Formulation'].notna()]
for col in min_data.columns[1:]:
    min_data[col] = pd.to_numeric(min_data[col], errors='coerce')
integrated['Minerals'] = min_data

vit = raw_data['Vitamins'].iloc[2:].reset_index(drop=True)
vit.columns = ['Formulation', 'VitA', 'VitE', 'VitK', 'VitC', 'VitB1', 'VitB2', 'VitB3', 'VitB5', 'VitB6', 'VitB7', 'VitB9']
vit = vit[vit['Formulation'].notna()]
for col in vit.columns[1:]:
    vit[col] = pd.to_numeric(vit[col], errors='coerce')
integrated['Vitamins'] = vit

tex = raw_data['Texture Analysis'].iloc[1:].reset_index(drop=True)
tex.columns = ['Formulation', 'Hardness_g', 'Firmness_g', 'Strength_g']
tex = tex[tex['Formulation'].notna()]
for col in tex.columns[1:]:
    tex[col] = pd.to_numeric(tex[col], errors='coerce')
integrated['Texture'] = tex

ph_data = raw_data['pH'].iloc[1:].reset_index(drop=True)
ph_data.columns = ['Formulation', 'pH']
ph_data = ph_data[ph_data['Formulation'].notna()]
ph_data['pH'] = pd.to_numeric(ph_data['pH'], errors='coerce')
integrated['pH'] = ph_data

col_data = raw_data['Colour Profile Analysis'].iloc[1:].reset_index(drop=True)
col_data.columns = ['Formulation', 'L_star', 'a_star', 'b_star']
col_data = col_data[col_data['Formulation'].notna()]
for col_name in ['L_star', 'a_star', 'b_star']:
    col_data[col_name] = pd.to_numeric(col_data[col_name], errors='coerce')
col_data['Chroma'] = np.sqrt(col_data['a_star']**2 + col_data['b_star']**2)
integrated['Color'] = col_data

sens = raw_data['Sensory Evaluation'].iloc[0:].reset_index(drop=True)
sens.columns = ['Formulation', 'Appearance', 'Colour', 'Texture', 'Flavour', 'Aroma', 'Overall_Accept']
sens = sens[sens['Formulation'].notna()]
for col in sens.columns[1:]:
    sens[col] = pd.to_numeric(sens[col], errors='coerce')
integrated['Sensory'] = sens

phyt = raw_data['Phytochemical and Antioxidant '].iloc[0:].reset_index(drop=True)
phyt.columns = ['Formulation', 'TPC_mgGAE', 'TFC_mgQE', 'DPPH_pct']
phyt = phyt[phyt['Formulation'].notna()]
for col in phyt.columns[1:]:
    phyt[col] = pd.to_numeric(phyt[col], errors='coerce')
integrated['Phytochemical'] = phyt

print('✓ All sheets integrated')


INTEGRATING DATA...
✓ All sheets integrated


In [5]:
# Master Dataset
master_df = integrated['Proximate'].copy()

for key in ['Minerals', 'Vitamins', 'Texture', 'pH', 'Color', 'Sensory', 'Phytochemical']:
    merge_cols = [col for col in integrated[key].columns if col != 'Formulation']
    master_df = master_df.merge(integrated[key][['Formulation'] + merge_cols], on='Formulation', how='outer')

master_df['Formulation_Name'] = master_df['Formulation'].map(formulation_map)

print(f'\n✓ MASTER DATASET:')
print(f'  Shape: {master_df.shape}')
print(f'  Observations: {len(master_df)}')
print(f'  Parameters: {master_df.shape[1] - 2}')

master_df.to_csv('outputs/phase_01_discovery/01_master_dataset.csv', index=False)
print('✓ Master dataset saved')


✓ MASTER DATASET:
  Shape: (4, 46)
  Observations: 4
  Parameters: 44
✓ Master dataset saved


# PHASE 2: EXPERIMENTAL DESIGN

In [6]:
print('\n' + '='*100)
print('PHASE 2: EXPERIMENTAL DESIGN')
print('='*100)

design_list = []
for form in ['A', 'B', 'C', 'D']:
    form_data = master_df[master_df['Formulation'] == form]
    design_list.append({
        'Code': form,
        'Name': formulation_map[form],
        'N': len(form_data)
    })

design_df = pd.DataFrame(design_list)
design_df.to_csv('outputs/phase_02_design/01_experimental_design.csv', index=False)

print('\nDESIGN SUMMARY:')
print(design_df.to_string(index=False))
print(f'\n✓ Total Observations: {len(master_df)}')
print('✓ Design: Completely Randomized Design (CRD)')


PHASE 2: EXPERIMENTAL DESIGN

DESIGN SUMMARY:
Code               Name  N
   A         Corn Flour  1
   B         Oats Flour  1
   C         Rice Flour  1
   D Puffed Rice Powder  1

✓ Total Observations: 4
✓ Design: Completely Randomized Design (CRD)


# PHASE 3: EXPLORATORY DATA ANALYSIS

In [7]:
print('\n' + '='*100)
print('PHASE 3: EXPLORATORY DATA ANALYSIS')
print('='*100)

key_params = ['Energy_kcal', 'Protein_pct', 'Fiber_g', 'Hardness_g', 'Overall_Accept', 'TPC_mgGAE']
stats_list = []

for param in key_params:
    for form in ['A', 'B', 'C', 'D']:
        data = master_df[master_df['Formulation'] == form][param].dropna()
        if len(data) > 0:
            stats_list.append({
                'Formulation': formulation_map[form],
                'Parameter': param,
                'Mean': data.mean(),
                'SD': data.std(),
                'Min': data.min(),
                'Max': data.max()
            })

stats_df = pd.DataFrame(stats_list)
stats_df.to_csv('outputs/phase_03_eda/01_descriptive_statistics.csv', index=False)
print('✓ Descriptive statistics: SAVED')


PHASE 3: EXPLORATORY DATA ANALYSIS
✓ Descriptive statistics: SAVED


In [8]:
# VIZ-01: Proximate Radar
prox_cols = ['Carb_pct', 'Protein_pct', 'Fat_pct', 'Moisture_g', 'Ash_g', 'Fiber_g']
fig1 = make_subplots(rows=2, cols=2, specs=[[{'type': 'scatterpolar'}, {'type': 'scatterpolar'}], [{'type': 'scatterpolar'}, {'type': 'scatterpolar'}]], subplot_titles=['Corn Flour', 'Oats Flour', 'Rice Flour', 'Puffed Rice'])

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for form, pos in zip(['A', 'B', 'C', 'D'], positions):
    form_data = master_df[master_df['Formulation'] == form][prox_cols].mean()
    fig1.add_trace(go.Scatterpolar(r=form_data.values, theta=prox_cols, fill='toself', name=formulation_map[form], marker=dict(color=formulation_colors[form], size=8), fillcolor=formulation_colors[form], opacity=0.6), row=pos[0], col=pos[1])

fig1.update_layout(title_text='VIZ-01: PROXIMATE RADAR', height=1000, showlegend=False, title_x=0.5)
fig1.write_html('outputs/phase_03_eda/viz_01_proximate_radar.html')
print('✓ VIZ-01 saved')

✓ VIZ-01 saved


In [9]:
# VIZ-02: Mineral Heatmap
mineral_cols = ['Na_mg', 'K_mg', 'Ca_mg', 'Zn_mg', 'Fe_mg', 'P_mg', 'I_mg', 'Mg_mg', 'Cu_mg']
mineral_summary = master_df.groupby('Formulation')[mineral_cols].mean()
mineral_summary.index = [formulation_map[f] for f in mineral_summary.index]
mineral_norm = mineral_summary.div(mineral_summary.max(axis=0), axis=1)

fig2 = go.Figure(data=go.Heatmap(z=mineral_norm.values, x=mineral_norm.columns, y=mineral_norm.index, colorscale='YlGnBu', text=np.round(mineral_summary.values, 2), texttemplate='%{text}', textfont={'size': 10}))
fig2.update_layout(title='VIZ-02: MINERAL HEATMAP', height=500, title_x=0.5)
fig2.write_html('outputs/phase_03_eda/viz_02_mineral_heatmap.html')
print('✓ VIZ-02 saved')

✓ VIZ-02 saved


In [10]:
# VIZ-03: Sensory Radar
sensory_cols = ['Appearance', 'Colour', 'Texture', 'Flavour', 'Aroma', 'Overall_Accept']
sensory_summary = master_df.groupby('Formulation')[sensory_cols].mean()

fig3 = go.Figure()
for form in ['A', 'B', 'C', 'D']:
    form_data = sensory_summary.loc[form]
    fig3.add_trace(go.Scatterpolar(r=form_data.values, theta=sensory_cols, fill='toself', name=formulation_map[form], marker=dict(color=formulation_colors[form], size=8), fillcolor=formulation_colors[form], opacity=0.4))

fig3.update_layout(title='VIZ-03: SENSORY PROFILES', polar=dict(radialaxis=dict(visible=True, range=[0, 10])), height=700, title_x=0.5)
fig3.write_html('outputs/phase_03_eda/viz_03_sensory_radar.html')
print('✓ VIZ-03 saved')

✓ VIZ-03 saved


In [11]:
# VIZ-04: Texture Distributions
texture_cols = ['Hardness_g', 'Firmness_g', 'Strength_g']
fig4 = make_subplots(rows=1, cols=3, subplot_titles=texture_cols, specs=[[{'type': 'box'}, {'type': 'box'}, {'type': 'box'}]])

for idx, col in enumerate(texture_cols):
    for form in ['A', 'B', 'C', 'D']:
        form_data = master_df[master_df['Formulation'] == form][col].dropna()
        fig4.add_trace(go.Box(y=form_data, name=formulation_map[form], marker=dict(color=formulation_colors[form]), showlegend=(idx==0)), row=1, col=idx+1)

fig4.update_layout(title_text='VIZ-04: TEXTURE DISTRIBUTIONS', height=600, title_x=0.5)
fig4.write_html('outputs/phase_03_eda/viz_04_texture_boxes.html')
print('✓ VIZ-04 saved')

✓ VIZ-04 saved


In [12]:
# VIZ-05: Antioxidant Analysis
antioxidant_cols = ['TPC_mgGAE', 'TFC_mgQE', 'DPPH_pct']
fig5 = make_subplots(rows=1, cols=3, subplot_titles=antioxidant_cols)

for idx, measure in enumerate(antioxidant_cols):
    for form in ['A', 'B', 'C', 'D']:
        form_data = master_df[master_df['Formulation'] == form][measure].dropna()
        fig5.add_trace(go.Box(y=form_data, name=formulation_map[form], marker=dict(color=formulation_colors[form]), showlegend=(idx==0)), row=1, col=idx+1)

fig5.update_layout(title_text='VIZ-05: ANTIOXIDANT ACTIVITY', height=550, title_x=0.5)
fig5.write_html('outputs/phase_03_eda/viz_05_antioxidant_boxes.html')
print('✓ VIZ-05 saved')

✓ VIZ-05 saved


# PHASE 4: STATISTICAL ANALYSIS

In [13]:
print('\n' + '='*100)
print('PHASE 4: STATISTICAL ANALYSIS')
print('='*100)

test_columns = ['Energy_kcal', 'Protein_pct', 'Fiber_g', 'Hardness_g', 'Overall_Accept', 'TPC_mgGAE']

# Normality Tests
print('\nNORMALITY TESTING:')
normality_results = {}
for col in test_columns:
    data = master_df[col].dropna()
    shapiro_stat, shapiro_p = stats.shapiro(data)
    ks_stat, ks_p = stats.kstest(data, 'norm', args=(data.mean(), data.std()))
    jb_stat, jb_p = stats.jarque_bera(data)
    nt_stat, nt_p = stats.normaltest(data)

    normality_results[col] = {'shapiro_p': shapiro_p, 'ks_p': ks_p, 'jb_p': jb_p, 'nt_p': nt_p}
    consensus = 'Normal' if shapiro_p > 0.05 else 'Non-Normal'
    print(f'{col:<20} Shapiro p={shapiro_p:.4f} | {consensus}')


PHASE 4: STATISTICAL ANALYSIS

NORMALITY TESTING:
Energy_kcal          Shapiro p=0.0020 | Non-Normal
Protein_pct          Shapiro p=0.8823 | Normal
Fiber_g              Shapiro p=0.1613 | Normal
Hardness_g           Shapiro p=0.9425 | Normal
Overall_Accept       Shapiro p=0.6046 | Normal
TPC_mgGAE            Shapiro p=0.6368 | Normal


In [14]:
# Homogeneity Tests
print('\nHOMOGENEITY TESTING:')
homogeneity_results = {}
for col in test_columns:
    groups = [master_df[master_df['Formulation'] == form][col].dropna().values for form in ['A', 'B', 'C', 'D']]
    levene_stat, levene_p = stats.levene(*groups)
    bartlett_stat, bartlett_p = stats.bartlett(*groups)
    bf_stat, bf_p = stats.levene(*groups, center='median')

    homogeneity_results[col] = {'levene_p': levene_p, 'bartlett_p': bartlett_p, 'bf_p': bf_p}
    homo = 'Homogeneous' if levene_p > 0.05 else 'Heterogeneous'
    print(f'{col:<20} Levene p={levene_p:.4f} | {homo}')


HOMOGENEITY TESTING:
Energy_kcal          Levene p=nan | Heterogeneous
Protein_pct          Levene p=nan | Heterogeneous
Fiber_g              Levene p=nan | Heterogeneous
Hardness_g           Levene p=nan | Heterogeneous
Overall_Accept       Levene p=nan | Heterogeneous
TPC_mgGAE            Levene p=nan | Heterogeneous


In [15]:
# ANOVA Analysis
print('\nANOVA ANALYSIS:')
anova_results = {}
for col in test_columns:
    groups = [master_df[master_df['Formulation'] == form][col].dropna().values for form in ['A', 'B', 'C', 'D']]
    f_stat, f_pval = stats.f_oneway(*groups)
    h_stat, h_pval = stats.kruskal(*groups)

    anova_results[col] = {'anova_f': f_stat, 'anova_p': f_pval, 'kw_h': h_stat, 'kw_p': h_pval}
    sig = '***' if f_pval < 0.001 else '**' if f_pval < 0.01 else '*' if f_pval < 0.05 else 'ns'
    print(f'{col:<20} F={f_stat:.3f} p={f_pval:.4f} {sig}')


ANOVA ANALYSIS:
Energy_kcal          F=nan p=nan ns
Protein_pct          F=nan p=nan ns
Fiber_g              F=nan p=nan ns
Hardness_g           F=nan p=nan ns
Overall_Accept       F=nan p=nan ns
TPC_mgGAE            F=nan p=nan ns


In [16]:
# Save Statistical Summary
stat_summary = []
for col in test_columns:
    ar = anova_results[col]
    stat_summary.append({
        'Parameter': col,
        'ANOVA_F': round(ar['anova_f'], 3),
        'ANOVA_p': round(ar['anova_p'], 4),
        'Significant': 'Yes' if ar['anova_p'] < 0.05 else 'No',
        'KW_H': round(ar['kw_h'], 3),
        'KW_p': round(ar['kw_p'], 4)
    })

stat_summary_df = pd.DataFrame(stat_summary)
stat_summary_df.to_csv('outputs/phase_04_statistics/01_statistical_summary.csv', index=False)
print('\n✓ Statistical summary saved')


✓ Statistical summary saved


# PHASE 5: MULTIVARIATE ANALYSIS

In [17]:
print('\n' + '='*100)
print('PHASE 5: MULTIVARIATE ANALYSIS')
print('='*100)

analysis_cols = [col for col in master_df.columns if col not in ['Formulation', 'Formulation_Name']]
X = master_df[analysis_cols].dropna()
X_scaled = StandardScaler().fit_transform(X)
formulations = master_df.loc[X.index, 'Formulation'].values

# PCA
pca = PCA()
pca.fit(X_scaled)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= 0.95) + 1
pca = PCA(n_components=min(n_components, X_scaled.shape[1]-1))
X_pca = pca.fit_transform(X_scaled)

print(f'\nPCA Results:')
print(f'  Components (95% variance): {pca.n_components}')
print(f'  Total variance: {pca.explained_variance_ratio_.sum()*100:.2f}%')


PHASE 5: MULTIVARIATE ANALYSIS

PCA Results:
  Components (95% variance): 2
  Total variance: 100.00%


In [18]:
# PCA Visualization
fig_pca = go.Figure()
for form in ['A', 'B', 'C', 'D']:
    mask = formulations == form
    fig_pca.add_trace(go.Scatter(x=X_pca[mask, 0], y=X_pca[mask, 1], mode='markers', name=formulation_map[form], marker=dict(size=12, color=formulation_colors[form], opacity=0.8)))

fig_pca.update_layout(title='VIZ-06: PCA SCORES PLOT', xaxis_title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', yaxis_title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', height=700, title_x=0.5)
fig_pca.write_html('outputs/phase_05_multivariate/viz_01_pca_2d.html')
print('✓ PCA plot saved')

✓ PCA plot saved


In [19]:
# Correlation Matrix
corr_matrix = master_df[analysis_cols].corr(method='pearson')
fig_corr = go.Figure(data=go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index, colorscale='RdBu', zmid=0, zmin=-1, zmax=1))
fig_corr.update_layout(title='VIZ-07: CORRELATION MATRIX', height=1000, width=1200, title_x=0.5)
fig_corr.write_html('outputs/phase_05_multivariate/viz_02_correlation.html')
corr_matrix.to_csv('outputs/phase_05_multivariate/01_correlation_matrix.csv')
print('✓ Correlation matrix saved')

✓ Correlation matrix saved


# PHASE 8: MACHINE LEARNING

In [31]:
print('\n' + '='*100)
print('PHASE 8: MACHINE LEARNING')
print('='*100)

ml_target = 'Overall_Accept'
ml_features = [col for col in analysis_cols if col != ml_target]

X_ml = master_df[ml_features].dropna()
y_ml = master_df.loc[X_ml.index, ml_target]

X_ml_scaled = StandardScaler().fit_transform(X_ml)
X_train, X_test, y_train, y_test = train_test_split(X_ml_scaled, y_ml, test_size=0.2, random_state=42)

print(f'\nML Data Prepared:')
print(f'  Training: {X_train.shape[0]} samples')
print(f'  Testing: {X_test.shape[0]} samples')
print(f'  Features: {X_train.shape[1]}')


PHASE 8: MACHINE LEARNING

ML Data Prepared:
  Training: 2 samples
  Testing: 1 samples
  Features: 43


In [32]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (a=1.0)': Ridge(alpha=1.0),
    'Lasso (a=0.01)': Lasso(alpha=0.01, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.01, max_iter=5000),
    'KNN (k=1)': KNeighborsRegressor(n_neighbors=1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
}

model_performance = {}
trained_models = {}

print(f'\n{"Model":<25} {"Train R2":<12} {"Test R2":<12} {"RMSE":<12} {"MAE":<12}')
print('-'*73)

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)

    model_performance[name] = {'train_r2': train_r2, 'test_r2': test_r2, 'rmse': rmse, 'mae': mae}
    print(f'{name:<25} {train_r2:<12.4f} {test_r2:<12.4f} {rmse:<12.4f} {mae:<12.4f}')

best_model_name = max(model_performance, key=lambda x: model_performance[x]['test_r2'])
print(f'\n✓ Best Model: {best_model_name}')


Model                     Train R2     Test R2      RMSE         MAE         
-------------------------------------------------------------------------
Linear Regression         1.0000       nan          0.2128       0.2128      
Ridge (a=1.0)             0.9996       nan          0.2120       0.2120      
Lasso (a=0.01)            0.9978       nan          0.1547       0.1547      
ElasticNet                0.9994       nan          0.1736       0.1736      
KNN (k=1)                 1.0000       nan          0.3500       0.3500      
Random Forest             0.7680       nan          0.1995       0.1995      
Gradient Boosting         1.0000       nan          0.1966       0.1966      
XGBoost                   1.0000       nan          0.0010       0.0010      
LightGBM                  -0.0000      nan          0.1750       0.1750      

✓ Best Model: Linear Regression


In [33]:
# Model Performance Visualization
perf_df = pd.DataFrame(model_performance).T.sort_values('test_r2', ascending=False)
perf_df.to_csv('outputs/phase_08_ml/01_model_performance.csv')

fig_ml = make_subplots(rows=2, cols=2, subplot_titles=['Test R2', 'Test RMSE', 'Test MAE', 'Train R2'])

fig_ml.add_trace(go.Bar(y=perf_df.index, x=perf_df['test_r2'], orientation='h', marker_color='#FF6B6B', showlegend=False), row=1, col=1)
fig_ml.add_trace(go.Bar(y=perf_df.index, x=perf_df['rmse'], orientation='h', marker_color='#4ECDC4', showlegend=False), row=1, col=2)
fig_ml.add_trace(go.Bar(y=perf_df.index, x=perf_df['mae'], orientation='h', marker_color='#45B7D1', showlegend=False), row=2, col=1)
fig_ml.add_trace(go.Bar(y=perf_df.index, x=perf_df['train_r2'], orientation='h', marker_color='#FFA07A', showlegend=False), row=2, col=2)

fig_ml.update_layout(title_text='VIZ-08: ML MODEL COMPARISON', height=900, title_x=0.5)
fig_ml.write_html('outputs/phase_08_ml/viz_01_model_comparison.html')
print('✓ Model comparison saved')

✓ Model comparison saved


# PHASE 10: QUALITY INDICES

In [34]:
print('\n' + '='*100)
print('PHASE 10: QUALITY INDICES')
print('='*100)

def norm_01(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-8)

master_df_idx = master_df.copy()

# NQI
nqi_cols = ['Protein_pct', 'Fiber_g', 'Ash_g']
nqi_data = master_df_idx[nqi_cols].fillna(master_df_idx[nqi_cols].mean())
master_df_idx['NQI'] = (norm_01(nqi_data['Protein_pct']) + norm_01(nqi_data['Fiber_g']) + norm_01(nqi_data['Ash_g'])) / 3 * 100

# AOI
aoi_cols = ['TPC_mgGAE', 'TFC_mgQE', 'DPPH_pct']
aoi_data = master_df_idx[aoi_cols].fillna(master_df_idx[aoi_cols].mean())
master_df_idx['AOI'] = (norm_01(aoi_data['TPC_mgGAE']) + norm_01(aoi_data['TFC_mgQE']) + norm_01(aoi_data['DPPH_pct'])) / 3 * 100

# MDI
mineral_idx = ['Ca_mg', 'Fe_mg', 'Mg_mg', 'Zn_mg']
mdi_data = master_df_idx[mineral_idx].fillna(master_df_idx[mineral_idx].mean())
master_df_idx['MDI'] = (norm_01(mdi_data['Ca_mg']) + norm_01(mdi_data['Fe_mg']) + norm_01(mdi_data['Mg_mg']) + norm_01(mdi_data['Zn_mg'])) / 4 * 100

# TQI
tqi_cols = ['Hardness_g', 'Firmness_g']
tqi_data = master_df_idx[tqi_cols].fillna(master_df_idx[tqi_cols].mean())
master_df_idx['TQI'] = (norm_01(tqi_data['Hardness_g']) + norm_01(tqi_data['Firmness_g'])) / 2 * 100

# SAI
sai_cols = ['Appearance', 'Colour', 'Texture', 'Flavour', 'Aroma', 'Overall_Accept']
sai_data = master_df_idx[sai_cols].fillna(master_df_idx[sai_cols].mean())
master_df_idx['SAI'] = sai_data[sai_cols].mean(axis=1) / 10 * 100

# OPEI
master_df_idx['OPEI'] = (master_df_idx['NQI'] * 0.25 + master_df_idx['AOI'] * 0.25 + master_df_idx['MDI'] * 0.15 + master_df_idx['TQI'] * 0.15 + master_df_idx['SAI'] * 0.20)

idx_cols = ['NQI', 'AOI', 'MDI', 'TQI', 'SAI', 'OPEI']
idx_summary = master_df_idx.groupby('Formulation')[idx_cols].mean()
idx_summary.index = [formulation_map[f] for f in idx_summary.index]

print('\nQUALITY INDEX RANKINGS:')
print(idx_summary.round(2))

idx_summary.to_csv('outputs/phase_10_indices/01_quality_indices.csv')

opei_rank = idx_summary['OPEI'].sort_values(ascending=False)
print('\nOPEI RANKING:')
for rank, (form, score) in enumerate(opei_rank.items(), 1):
    print(f'  {rank}. {form:<20} {score:.2f}')


PHASE 10: QUALITY INDICES

QUALITY INDEX RANKINGS:
                      NQI    AOI     MDI     TQI    SAI   OPEI
Corn Flour          18.51   0.00    0.00  100.00  84.67  36.56
Oats Flour          66.67  85.19  100.00   68.72  80.85  79.44
Rice Flour          38.16  16.64   20.38   34.57  82.98  38.54
Puffed Rice Powder  50.90  86.57   34.07    0.00  80.47  55.57

OPEI RANKING:
  1. Oats Flour           79.44
  2. Puffed Rice Powder   55.57
  3. Rice Flour           38.54
  4. Corn Flour           36.56


In [35]:
# Quality Indices Heatmap
fig_idx = go.Figure(data=go.Heatmap(z=idx_summary.values, x=idx_summary.columns, y=idx_summary.index, colorscale='YlGnBu', text=np.round(idx_summary.values, 1), texttemplate='%{text}', textfont={'size': 12}))
fig_idx.update_layout(title='VIZ-09: QUALITY INDICES HEATMAP', height=500, title_x=0.5)
fig_idx.write_html('outputs/phase_10_indices/viz_01_indices_heatmap.html')
print('✓ Quality indices heatmap saved')

✓ Quality indices heatmap saved


# FINAL: AUTOMATIC ZIP & DOWNLOAD

In [36]:
print('\n' + '='*100)
print('CREATING OUTPUT ARCHIVE')
print('='*100)

zip_filename = 'Gummy_Formulation_Complete_Analysis_Outputs.zip'
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', 'outputs')
zip_size = os.path.getsize(zip_filename) / (1024*1024)

print(f'\n✓ ZIP archive: {zip_filename}')
print(f'✓ Size: {zip_size:.2f} MB')
print(f'\n✓ CONTENTS:')
print('  Phase 1: Master Dataset CSV')
print('  Phase 2: Experimental Design CSV')
print('  Phase 3: EDA (5 Visualizations HTML)')
print('  Phase 4: Statistics Summary CSV')
print('  Phase 5: Multivariate (2 Visualizations HTML + CSV)')
print('  Phase 8: ML Comparison (1 Visualization HTML + CSV)')
print('  Phase 10: Quality Indices (1 Visualization HTML + CSV)')
print(f'\n✓ Total Output Files: 20+')
print('\n✓ READY FOR DOWNLOAD!')


CREATING OUTPUT ARCHIVE

✓ ZIP archive: Gummy_Formulation_Complete_Analysis_Outputs.zip
✓ Size: 11.49 MB

✓ CONTENTS:
  Phase 1: Master Dataset CSV
  Phase 2: Experimental Design CSV
  Phase 3: EDA (5 Visualizations HTML)
  Phase 4: Statistics Summary CSV
  Phase 5: Multivariate (2 Visualizations HTML + CSV)
  Phase 8: ML Comparison (1 Visualization HTML + CSV)
  Phase 10: Quality Indices (1 Visualization HTML + CSV)

✓ Total Output Files: 20+

✓ READY FOR DOWNLOAD!


In [37]:
try:
    from google.colab import files
    files.download(zip_filename)
    print(f'✓ Download initiated: {zip_filename}')
except:
    print(f'✓ Archive ready at: {zip_filename}')

print('\n' + '='*100)
print('🎉 ANALYSIS COMPLETE')
print('='*100)
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('Status: ✓ SUCCESS')
print('\nEXTRACT ZIP & EXPLORE:')
print('  • Open .html in browser (interactive plots)')
print('  • Open .csv in Excel (data)')
print('  • Share with stakeholders!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download initiated: Gummy_Formulation_Complete_Analysis_Outputs.zip

🎉 ANALYSIS COMPLETE
Timestamp: 2026-06-27 12:08:44
Status: ✓ SUCCESS

EXTRACT ZIP & EXPLORE:
  • Open .html in browser (interactive plots)
  • Open .csv in Excel (data)
  • Share with stakeholders!
